In [ ]:
import random
from turtledemo.nim import randommove
!pip install -q pinecone-client

In [1]:
import os
from dotenv import load_dotenv , find_dotenv

In [10]:
load_dotenv(dotenv_path='conf/.env')

True

In [18]:
from pinecone import Pinecone
pc  = Pinecone()
#pc  = Pinecone(api_key='your api key')
pc.list_indexes()

{'indexes': []}

In [17]:
import ipywidgets as widgets

In [19]:
pc.list_collections()

{'collections': []}

In [20]:
pc.list_indexes()

{'indexes': []}

In [24]:
from pinecone import ServerlessSpec
index_name='langchain'
if index_name not in pc.list_indexes().names():
    print('Index not found , creating afresh')
    pc.create_index(name=index_name,
                    dimension=1536,
                    metric='cosine',
                    spec=ServerlessSpec(
                        cloud="aws",
                        region="us-east-1"
                        )
                    )
    print(f'{index_name} index created')
else:
    print('Index found')

Index not found , creating afresh
langchain index created


In [23]:
index_name='langchain'
if index_name in pc.list_indexes().names():
    print('Index found , deleting')
    pc.delete_index(name=index_name)
    print(f'{index_name} index deleted')
else:
    print('Index not found')

Index not found


In [26]:
index=pc.Index(name=index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

In [ ]:
import random
vectors= [[random.random() for _ in range(1536)] for _ in range(5)]
print(vectors)
ids= list('abcde')
index_name='langchain'
index=pc.Index(name=index_name)
index.upsert(vectors=zip(ids,vectors))

In [30]:
index.fetch(ids=['c','d'])



{'namespace': '',
 'usage': {'read_units': 1},
 'vectors': {'c': {'id': 'c',
                   'values': [0.610789955,
                              0.601486683,
                              0.471878976,
                              0.751547754,
                              0.62857008,
                              0.279290527,
                              0.769229412,
                              0.449591815,
                              0.185841888,
                              0.543962538,
                              0.64626509,
                              0.636301875,
                              0.204364,
                              0.61799556,
                              0.378661364,
                              0.78128016,
                              0.83044225,
                              0.171524167,
                              0.855319083,
                              0.0793902799,
                              0.843551219,
                           

In [36]:
##not working index.upsert(vectors=zip('c',[0.5]*1536))
index.upsert(vectors=[('c',[0.5]*1536)])

{'upserted_count': 1}

In [38]:
index.fetch(ids=['c'])

{'namespace': '',
 'usage': {'read_units': 1},
 'vectors': {'c': {'id': 'c',
                   'values': [0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
                              0.5,
             

In [39]:
index.delete(ids=['b','c'])

{}

In [40]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 3}},
 'total_vector_count': 3}

In [41]:
index.fetch(ids=['index_not_in_db'])

{'namespace': '', 'usage': {'read_units': 1}, 'vectors': {}}

In [46]:
import random
item_you_need_to_query=[random.random() for _ in range(1536)]
index.query(vector=item_you_need_to_query,top_k=2, include_values=False)

{'matches': [{'id': 'd', 'score': 0.762243, 'values': []},
             {'id': 'e', 'score': 0.756083131, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 5}}

NAMESPACES

In [47]:
vectors= [[random.random() for _ in range(1536)] for _ in range(5)]
ids= list('abcde')
index_name='langchain'
index=pc.Index(name=index_name)
index.upsert(vectors=zip(ids,vectors))

{'upserted_count': 5}

In [48]:
vectors= [[random.random() for _ in range(1536)] for _ in range(3)]
ids= list('xyz')
index_name='langchain'
index=pc.Index(name=index_name)
index.upsert(vectors=zip(ids,vectors),namespace='first-namespace')

{'upserted_count': 3}

In [49]:
vectors= [[random.random() for _ in range(1536)] for _ in range(4)]
ids= list('lmno')
index_name='langchain'
index=pc.Index(name=index_name)
index.upsert(vectors=zip(ids,vectors),namespace='second-namespace')

{'upserted_count': 4}

In [50]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 5},
                'first-namespace': {'vector_count': 3},
                'second-namespace': {'vector_count': 4}},
 'total_vector_count': 12}

In [ ]:
#By default it gets only from default namespace. If you want from another you have to specify
index.fetch(ids=['x','d'])

In [52]:
index.delete(delete_all=True,namespace='first-namespace')

{}

In [53]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 5},
                'second-namespace': {'vector_count': 4}},
 'total_vector_count': 9}